# Decorators in Python (with Trading API Examples)

This notebook is a **from-scratch to intermediate/advanced** guide to Python **decorators**.

You already know OOP; here we focus on **what decorators are, how they work, and how to
use them cleanly**, using trading-related examples:

- Decorating **message handlers** for REST and WebSocket endpoints
- Centralized **error handling**, logging, and retry logic
- Adding **authorization**, timing, and caching behavior around functions

We will build up gradually:

1. **Core ideas**: functions as objects, first-class functions
2. **Basic decorators** without arguments
3. **Decorators with arguments** (configurable behavior)
4. **Preserving metadata with `functools.wraps`**
5. **Error-handling and logging decorators for REST/WS callbacks**
6. **Async decorators** for WebSocket-style coroutines
7. **Stacking multiple decorators and ordering concerns
8. **Best practices, pitfalls, and when not to use decorators

> Run and modify the code cells as you go. The goal is to make decorators a
> practical tool for structuring cross-cutting concerns in trading systems.

### Contents

- [1. Core Ideas: Functions as First-Class Objects](#1-core-ideas-functions-as-first-class-objects)
- [2. Basic Decorators (No Arguments)](#2-basic-decorators-no-arguments)
- [3. Decorators with Arguments](#3-decorators-with-arguments)
- [4. Error-Handling and Logging Decorators for REST / WebSocket Callbacks](#4-error-handling-and-logging-decorators-for-rest--websocket-callbacks)
- [5. Async Decorators for WebSocket Message Handlers](#5-async-decorators-for-websocket-message-handlers)
- [6. More Practical Decorator Patterns](#6-more-practical-decorator-patterns)
- [7. Best Practices, Pitfalls, and When Not to Use Decorators](#7-best-practices-pitfalls-and-when-not-to-use-decorators)

## 1. Core Ideas: Functions as First-Class Objects

Decorators build on one fundamental Python idea:

> **Functions are objects**: you can pass them around, store them, and return them
> from other functions.

This lets you write **higher-order functions** (functions that take or return other
functions), which is exactly what decorators are.

In [ ]:
# 1.1 Functions as values

def handle_rest_message(message: str) -> None:
    print("Handling REST message:", message)


def call_handler(handler, message: str) -> None:
    # `handler` is just a function we pass in
    handler(message)


call_handler(handle_rest_message, "order created")

## 2. Basic Decorators (No Arguments)

A **decorator** is a callable that:

- Takes a function `f` as input.
- Returns a **new function** that usually wraps `f` with extra behavior.

Syntax sugar:

```python
@my_decorator
def fn(...):
    ...
```

is equivalent to:

```python
def fn(...):
    ...

fn = my_decorator(fn)
```

We'll start with a simple **logging decorator** for message handlers.

In [ ]:
# 2.1 A simple logging decorator

from collections.abc import Callable
from typing import Any


def log_calls(func: Callable[..., Any]) -> Callable[..., Any]:
    """Log function calls and return values.

    This is a basic decorator without arguments.
    """

    def wrapper(*args: Any, **kwargs: Any) -> Any:
        print(f"[log_calls] Calling {func.__name__} with args={args}, kwargs={kwargs}")
        result = func(*args, **kwargs)
        print(f"[log_calls] {func.__name__} returned {result!r}")
        return result

    return wrapper


@log_calls
def parse_rest_message(message: str) -> dict[str, Any]:
    # Very toy parser for demo purposes
    return {"raw": message, "length": len(message)}


parsed = parse_rest_message("order: buy 100 AAPL")
print("Parsed:", parsed)


### 2.2 Wrapping message handlers

Decorators are especially useful for **message handlers** (REST, WebSocket, Kafka, etc.).

You often want all handlers to:

- Log incoming messages
- Catch and log errors
- Maybe measure latency

Without duplicating that code in every handler.

In [ ]:
# 2.3 A simple error-handling decorator for handlers

import json
from typing import Any, Dict


def simple_error_handler(func: Callable[..., Any]) -> Callable[..., Any]:
    """Catch exceptions in message handlers and return an error response.

    For REST, we might return a dict representing a JSON response.
    """

    def wrapper(message: str) -> Dict[str, Any]:
        try:
            return func(message)
        except Exception as e:
            print(f"[error_handler] Error in {func.__name__}: {e}")
            return {"status": "error", "error": str(e)}

    return wrapper


@simple_error_handler
def handle_rest_order(message: str) -> dict[str, Any]:
    # Pretend the message is JSON
    data = json.loads(message)  # may raise
    if data.get("qty", 0) <= 0:
        raise ValueError("qty must be positive")
    return {"status": "ok", "order_id": 123}


print(handle_rest_order('{"symbol": "AAPL", "qty": 100}'))
print(handle_rest_order('{"symbol": "AAPL", "qty": 0}'))  # triggers ValueError
print(handle_rest_order("not json at all"))  # triggers JSONDecodeError


## 3. Decorators with Arguments

Sometimes you want to **configure** a decorator:

- Control log level
- Toggle behavior (e.g. retry on/off)
- Pass context (e.g. service name, symbol filter)

Pattern:

1. Outer function takes the decorator arguments.
2. It returns the actual decorator, which takes `func`.
3. That decorator returns the `wrapper`.

We'll implement a **configurable retry decorator** for REST calls.

In [ ]:
# 3.1 Retry decorator with arguments

import time
from functools import wraps


def retry(max_attempts: int = 3, delay: float = 0.1):
    """Retry a function on exception up to `max_attempts`.

    This is a decorator *factory*: it returns the actual decorator.
    """

    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        @wraps(func)  # preserve metadata of func
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            attempt = 0
            while True:
                attempt += 1
                try:
                    return func(*args, **kwargs)
                except Exception as e:
                    print(f"[retry] {func.__name__} failed on attempt {attempt}: {e}")
                    if attempt >= max_attempts:
                        print("[retry] giving up")
                        raise
                    time.sleep(delay)

        return wrapper

    return decorator


_counter = {"failures": 0}


@retry(max_attempts=3, delay=0.05)
def flaky_rest_call() -> str:
    """Simulate a flaky REST call that fails twice before succeeding."""
    if _counter["failures"] < 2:
        _counter["failures"] += 1
        raise RuntimeError("temporary network error")
    return "success after retries"


print(flaky_rest_call())

### 3.2 Why `functools.wraps` matters

Without `@wraps(func)`, the `wrapper` would:

- Have the wrong `__name__`, `__doc__`, and other metadata.
- Confuse tools like debuggers, loggers, and documentation generators.

Always use `@wraps` for non-trivial decorators so that the wrapped function still
*looks like* the original to callers and tooling.

Next, we'll combine these ideas into a more realistic **error-handling master decorator**
for REST and WebSocket callbacks.

## 4. Error-Handling and Logging Decorators for REST / WebSocket Callbacks

We'll design a **master decorator** that can wrap different message-reading callbacks
(e.g. REST, WebSocket) with consistent behavior:

- Structured logging of input and output
- Exception capture and standardized error responses
- Optional retry for transient errors

We'll start with synchronous REST-style handlers, then show the async variant for WebSockets.

In [ ]:
# 4.1 Master decorator for REST message handlers

from functools import wraps
from typing import Callable, Any, Dict


def rest_handler(service: str):
    """Decorator factory for REST message handlers.

    Adds logging, error handling, and a standard response structure.
    """

    def decorator(func: Callable[..., Dict[str, Any]]) -> Callable[..., Dict[str, Any]]:
        @wraps(func)
        def wrapper(message: str) -> Dict[str, Any]:
            print(f"[{service}] Incoming REST message: {message}")
            try:
                payload = func(message)
                response = {"status": "ok", "payload": payload}
                print(f"[{service}] Handler {func.__name__} OK: {response}")
                return response
            except Exception as e:
                print(f"[{service}] Handler {func.__name__} ERROR: {e}")
                return {"status": "error", "error": str(e)}

        return wrapper

    return decorator


@rest_handler(service="order-service")
def create_order_handler(message: str) -> Dict[str, Any]:
    data = json.loads(message)
    if data.get("qty", 0) <= 0:
        raise ValueError("qty must be positive")
    # In real life you'd insert into a DB or call a broker API here
    return {"order_id": 42, "symbol": data["symbol"], "qty": data["qty"]}


print(create_order_handler('{"symbol": "AAPL", "qty": 100}'))
print(create_order_handler('{"symbol": "AAPL", "qty": 0}'))

### 4.2 Adding retry on top (stacking decorators)

You can **stack multiple decorators** on the same function:

```python
@decorator_a
@decorator_b
def fn(...):
    ...
```

is equivalent to `fn = decorator_a(decorator_b(fn))`.

Order matters. We'll add `@retry` on top of `@rest_handler` to implement a handler
that retries on transient errors before giving up.

In [ ]:
# 4.3 Stacking rest_handler and retry

_transient_counter = {"failures": 0}


@rest_handler(service="marketdata-service")
@retry(max_attempts=3, delay=0.05)
def fetch_last_price_handler(message: str) -> Dict[str, Any]:
    """Simulate a REST handler that fetches the last price and sometimes fails."""
    _transient_counter["failures"] += 1
    if _transient_counter["failures"] < 3:
        # Simulate transient failure
        raise RuntimeError("temporary DB error")
    data = json.loads(message)
    symbol = data["symbol"]
    # Fake price
    return {"symbol": symbol, "last_price": 123.45}


print(fetch_last_price_handler('{"symbol": "AAPL"}'))

Note: here `@retry` wraps the inner function **first**, and then `@rest_handler`
wraps the result. That means:

- `fetch_last_price_handler` (outermost) always returns the standardized REST response.
- The retry logic happens *inside* that, around the core handler.

If you reverse the order, the behavior changes (the retry would wrap the entire
REST response wrapper instead).

Next we will look at **async decorators** for WebSocket-style coroutines.

## 5. Async Decorators for WebSocket Message Handlers

Modern trading systems often use **async WebSocket clients** (e.g. for live market data).
Handlers are then defined as **`async def` coroutines**.

Decorators can wrap these as well, but the wrapper must also be `async` and use `await`.

We'll build:

- An `async_ws_handler` decorator for logging and error handling.
- A toy WebSocket message handler that decodes JSON and updates some in-memory state.

In [ ]:
# 5.1 Async WebSocket handler decorator

import asyncio
from typing import Awaitable, Callable


WsHandler = Callable[[str], Awaitable[None]]  # async handler taking a text message


def async_ws_handler(stream_name: str):
    """Decorator factory for async WebSocket message handlers.

    Adds logging and error handling around async callbacks.
    """

    def decorator(func: WsHandler) -> WsHandler:
        @wraps(func)
        async def wrapper(message: str) -> None:
            print(f"[{stream_name}] WS message: {message}")
            try:
                await func(message)
            except Exception as e:
                print(f"[{stream_name}] ERROR in {func.__name__}: {e}")

        return wrapper  # type: ignore[return-value]

    return decorator


# Shared in-memory state for demo purposes
_latest_prices: dict[str, float] = {}


@async_ws_handler(stream_name="ticker-stream")
async def ws_price_handler(message: str) -> None:
    data = json.loads(message)
    symbol = data["symbol"]
    price = float(data["price"])
    _latest_prices[symbol] = price
    print(f"Updated latest price for {symbol} -> {price}")


async def demo_ws_flow() -> None:
    messages = [
        '{"symbol": "AAPL", "price": 100.1}',
        '{"symbol": "AAPL", "price": 100.2}',
        'not json',  # triggers error handling
        '{"symbol": "MSFT", "price": 200.5}',
    ]
    for msg in messages:
        await ws_price_handler(msg)


asyncio.run(demo_ws_flow())
print("Latest prices:", _latest_prices)


## 6. More Practical Decorator Patterns

Beyond logging and error handling, decorators are commonly used in trading systems for:

- **Authorization / authz**: ensure the caller/session has permission.
- **Timing / metrics**: record latency and call counts (e.g. for Prometheus).
- **Caching**: memoize expensive calls (e.g. reference data lookups).

We'll sketch a simple **timing decorator** and show how it can be layered with others.

In [ ]:
# 6.1 Timing decorator

import time
from functools import wraps


def timed(label: str):
    """Measure and print the execution time of a function."""

    def decorator(func: Callable[..., Any]) -> Callable[..., Any]:
        @wraps(func)
        def wrapper(*args: Any, **kwargs: Any) -> Any:
            start = time.perf_counter()
            try:
                return func(*args, **kwargs)
            finally:
                elapsed = time.perf_counter() - start
                print(f"[timed:{label}] {func.__name__} took {elapsed * 1000:.2f} ms")

        return wrapper

    return decorator


@timed("complex-risk-check")
@rest_handler(service="risk-service")
def risk_check_handler(message: str) -> Dict[str, Any]:
    # Fake expensive logic
    data = json.loads(message)
    time.sleep(0.05)
    if abs(data.get("notional", 0)) > 1_000_000:
        raise ValueError("Notional limit exceeded")
    return {"allowed": True}


print(risk_check_handler('{"notional": 500000}'))
print(risk_check_handler('{"notional": 1500000}'))

Here `@timed` wraps the entire REST handler, including logging/error handling from
`@rest_handler`. In performance-sensitive code, you might choose to time only the
"core" part; decorator order lets you control that.

Next, we'll summarize **best practices** and common pitfalls when using decorators.

## 7. Best Practices, Pitfalls, and When Not to Use Decorators

### 7.1 Best practices

- **Keep decorators small and focused**: one concern per decorator (logging, retry,
  timing, auth, etc.).
- **Always use `functools.wraps`** for non-trivial decorators.
- Prefer **decorator factories** for configurable behavior (`@retry(...)`, `@rest_handler(...)`).
- Document whether a decorator **changes return types** (e.g. wrapping in a response dict).

### 7.2 Common pitfalls

- **Losing metadata**: skipping `@wraps` breaks `__name__`, docstrings, and tooling.
- **Order confusion** when stacking: remember that decorators apply from bottom to top.
- **Over-decorating**: too many layers can make control flow hard to follow.
- Using decorators where a **plain helper function** would be clearer.

### 7.3 How to practice

- Wrap your existing REST/WS handlers with small decorators for logging and error handling.
- Introduce a configurable `@retry` for transient API calls.
- Add a `@timed` or metrics-reporting decorator and check latencies under load.

Once these patterns feel natural, decorators become a powerful tool to express
**cross-cutting concerns** cleanly, especially around message-handling code in
trading systems.